# 02.2 — Matrix Operations

**Phase:** 02 — Mathematics for ML

**Difficulty:** Beginner–Intermediate

**Priority:** Core — linear regression, neural networks, PCA all depend on this

**Status:** VERIFIED

---

## 1. What Are We Solving?

In the last notebook, you learned what vectors and matrices *are*. Now you'll learn what you can *do* with them.

The key operations — multiplication, inverse, and solving linear systems — are not abstract math. They are the literal computation that happens inside every ML model:
- **Matrix multiplication**: every forward pass in a neural network
- **Inverse / solving**: the normal equation that solves linear regression

## 2. Why Does This Matter?

Matrix multiplication is the engine of deep learning. A single neural network layer is just:
```
output = activation(input @ weights + bias)
```
That's one matrix multiply, one element-wise addition, and one activation function. Stack 100 layers, and you're doing 100 matrix multiplies.

The inverse and linear system solving power the **normal equation** for linear regression: `w = (XᵀX)⁻¹Xᵀy`. Understanding *why* this works gives you intuition for regularization, instability, and when gradient descent is preferred.

## 3. Prerequisites

- Unit 02.1 (Vectors & Matrices) — you should know shapes, dot products, and transpose
- Basic NumPy indexing

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- [ ] Multiply matrices and verify results by hand
- [ ] Explain why inner dimensions must match in matrix multiplication
- [ ] Compute a matrix inverse and understand when it doesn't exist (singular matrices)
- [ ] Solve linear systems using `np.linalg.solve`
- [ ] Explain how the normal equation solves linear regression
- [ ] Know when to use `solve` vs `inv` vs gradient descent

## 5. Mental Model

Think of matrix multiplication as a **transformation pipeline**. If matrix A rotates a vector and matrix B scales it, then `B @ A` rotates *then* scales. The order matters — matrix multiplication is not commutative.

Think of the **inverse** as an "undo" button. If matrix A rotates something 30°, then A⁻¹ rotates it back -30°. Multiplying A @ A⁻¹ gives you the identity — nothing happened.

Think of **solving Ax = b** as asking: "What input x, when transformed by A, gives me b?" The answer is x = A⁻¹b.

## 6. Core Concepts: Matrix Multiplication

Matrix multiplication combines two matrices into one. The key rule: **inner dimensions must match**.

```
A (m×n) @ B (n×p) = C (m×p)
        ↑   ↑
        must be equal
```

**How each element is computed:**
Each element `C[i,j]` is the dot product of row `i` from A and column `j` from B:

```
C[i,j] = A[i,0]*B[0,j] + A[i,1]*B[1,j] + ... + A[i,n-1]*B[n-1,j]

Example:  [1 2] @ [5 6] = [1*5+2*7, 1*6+2*8] = [19, 22]
          [3 4]   [7 8]   [3*5+4*7, 3*6+4*8]   [43, 50]
```

**In ML:** Matrix multiplication is how:
- A neural network layer transforms inputs: `X @ W`
- Multiple data points are processed in parallel (batch matrix multiply)
- Attention in transformers combines queries and keys: `Q @ K.T`

In [1]:
import numpy as np

# Matrix multiplication
A = np.array([[1, 2], [3, 4]])   # 2x2
B = np.array([[5, 6], [7, 8]])   # 2x2

print(f"A:\n{A}")
print(f"B:\n{B}")

C = A @ B
print(f"\nA @ B:\n{C}")

# Verify manually: C[0,0] = 1*5 + 2*7 = 19
print(f"C[0,0] = {C[0,0]} (expected 19)")
print(f"C[0,1] = {C[0,1]} (expected 22)")

A:
[[1 2]
 [3 4]]
B:
[[5 6]
 [7 8]]

A @ B:
[[19 22]
 [43 50]]
C[0,0] = 19 (expected 19)
C[0,1] = 22 (expected 22)


In [2]:
# Matrix multiplication with different shapes
# A is 2x3, B is 3x2 -> result is 2x2
A = np.array([[1, 2, 3], [4, 5, 6]])   # 2x3
B = np.array([[1, 2], [3, 4], [5, 6]]) # 3x2

print(f"A shape: {A.shape}, B shape: {B.shape}")
C = A @ B
print(f"A @ B shape: {C.shape}")
print(f"C:\n{C}")

A shape: (2, 3), B shape: (3, 2)
A @ B shape: (2, 2)
C:
[[22 28]
 [49 64]]


## 7. Matrix-Vector Multiplication

A matrix-vector multiply is a special case of matrix multiplication where the second "matrix" has just one column. The result is a vector.

```
W (m×n) @ x (n,) = output (m,)

Each output[i] = dot product of row i of W with x
               = W[i,0]*x[0] + W[i,1]*x[1] + ... + W[i,n-1]*x[n-1]
```

**This is literally a neural network layer.** The weight matrix W transforms the input vector x into a new representation. Adding a bias vector b shifts the result.

```
neural_net_layer(x) = activation(W @ x + b)
                     ↑      ↑     ↑   ↑
                   nonlin  matmul  add bias
```

In [3]:
# Matrix-vector multiplication (neural network layer)
# 3 input features -> 2 output features
W = np.array([[0.5, -0.2, 0.1],   # weights for output 1
              [0.3, 0.8, -0.4]])  # weights for output 2
x = np.array([1.0, 2.0, 3.0])     # input features
b = np.array([0.1, -0.1])         # bias

output = W @ x + b
print(f"W (2x3):\n{W}")
print(f"x: {x}")
print(f"b: {b}")
print(f"\noutput = W @ x + b: {output}")

# Verify manually: output[0] = 0.5*1 + (-0.2)*2 + 0.1*3 + 0.1 = 0.5 - 0.4 + 0.3 + 0.1 = 0.5
print(f"output[0] = {output[0]:.2f} (expected 0.50)")

W (2x3):
[[ 0.5 -0.2  0.1]
 [ 0.3  0.8 -0.4]]
x: [1. 2. 3.]
b: [ 0.1 -0.1]

output = W @ x + b: [0.5 0.6]
output[0] = 0.50 (expected 0.50)


## 8. Identity Matrix

The **identity matrix** `I` is the matrix equivalent of the number 1. Multiplying by it does nothing.

```
A @ I = A   (right-multiply by I)
I @ A = A   (left-multiply by I)

I₂ = [1 0]
     [0 1]
```

**Why it matters:**
- The inverse is defined in terms of I: `A @ A⁻¹ = I`
- The identity is the "do-nothing" transformation
- Adding `λI` (regularization) is like adding a small "do-nothing" component to prevent singular matrices

In [4]:
# Identity matrix
A = np.array([[1, 2], [3, 4]])
I = np.eye(2)

print(f"A:\n{A}")
print(f"I:\n{I}")
print(f"\nA @ I:\n{A @ I}")
print(f"I @ A:\n{I @ A}")
print(f"\nA @ I == A: {np.allclose(A @ I, A)}")

A:
[[1 2]
 [3 4]]
I:
[[1. 0.]
 [0. 1.]]

A @ I:
[[1. 2.]
 [3. 4.]]
I @ A:
[[1. 2.]
 [3. 4.]]

A @ I == A: True


## 9. Matrix Inverse

The **inverse** of a matrix `A` (written `A⁻¹`) is the matrix that "undoes" A's transformation.

```
A @ A⁻¹ = I   (A followed by its inverse = nothing)
A⁻¹ @ A = I   (same in reverse)
```

**Key rules:**
- Only **square** matrices can have inverses
- The matrix must be **non-singular** (determinant ≠ 0)
- If det(A) = 0, the matrix is **singular** — it collapses space into a lower dimension and can't be undone

**The intuition:** If matrix A rotates a vector 30°, then A⁻¹ rotates it back -30°. If A scales by 2, A⁻¹ scales by 0.5. The composition A @ A⁻¹ = I means "rotate then un-rotate" = nothing happened.

In [5]:
# Matrix inverse
A = np.array([[4, 7], [2, 6]])

A_inv = np.linalg.inv(A)
print(f"A:\n{A}")
print(f"\nA_inv:\n{A_inv}")

# Verify: A @ A_inv = I
print(f"\nA @ A_inv:\n{A @ A_inv}")
print(f"Is identity: {np.allclose(A @ A_inv, np.eye(2))}")

# Determinant (must be non-zero for invertible)
det = np.linalg.det(A)
print(f"\nDeterminant: {det:.3f}")

A:
[[4 7]
 [2 6]]

A_inv:
[[ 0.6 -0.7]
 [-0.2  0.4]]

A @ A_inv:
[[ 1.00000000e+00 -1.11022302e-16]
 [ 1.11022302e-16  1.00000000e+00]]
Is identity: True

Determinant: 10.000


In [6]:
# Singular matrix (not invertible)
singular = np.array([[1, 2], [2, 4]])  # rows are multiples
det = np.linalg.det(singular)
print(f"Determinant of singular matrix: {det:.6f}")

try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as e:
    print(f"LinAlgError: {e}")
    print("Singular matrices (det=0) have no inverse.")

Determinant of singular matrix: 0.000000
LinAlgError: Singular matrix
Singular matrices (det=0) have no inverse.


## 10. Solving Linear Systems

Given a matrix A and a vector b, find the vector x such that `Ax = b`.

```
Ax = b
A⁻¹(Ax) = A⁻¹b   (multiply both sides by A⁻¹)
Ix = A⁻¹b         (A⁻¹A = I)
x = A⁻¹b          (solution)
```

**In practice:** Don't compute `A⁻¹ @ b` directly — it's numerically unstable. Instead, use `np.linalg.solve(A, b)`, which solves the system without explicitly computing the inverse.

**When this shows up in ML:**
- The normal equation for linear regression: `w = (XᵀX)⁻¹Xᵀy`
- Solving for coefficients in least-squares problems
- Gaussian elimination in optimization

In [7]:
# Solve Ax = b
A = np.array([[3, 1], [1, 2]])
b = np.array([9, 8])

# Method 1: x = A^-1 b
x1 = np.linalg.inv(A) @ b
print(f"x = A^-1 b: {x1}")

# Method 2: np.linalg.solve (more stable)
x2 = np.linalg.solve(A, b)
print(f"np.linalg.solve: {x2}")

# Verify: A @ x should equal b
print(f"\nA @ x: {A @ x2}")
print(f"b: {b}")
print(f"Match: {np.allclose(A @ x2, b)}")

x = A^-1 b: [2. 3.]


np.linalg.solve: [2. 3.]

A @ x: [9. 8.]
b: [9 8]
Match: True


## 11. Experiment: Linear Regression with the Normal Equation

The **normal equation** gives us the optimal weights for linear regression in one step — no gradient descent needed.

```
w = (XᵀX)⁻¹ Xᵀy

Where:
  X = feature matrix (n_samples × n_features), with bias column of 1s
  y = target vector (n_samples,)
  w = optimal weights (n_features + 1,)
```

**Why this works:** The normal equation minimizes the sum of squared errors analytically. It finds the exact solution in one computation.

**When to use it vs. gradient descent:**
- Normal equation: fast for small datasets (< ~10K samples, < ~1K features)
- Gradient descent: necessary for large datasets (normal equation requires O(n³) matrix inverse)

In [8]:
# Linear regression with the normal equation
np.random.seed(42)

# Generate data: y = 3x1 - 2x2 + 5 + noise
n = 200
X = np.random.randn(n, 2)
true_w = np.array([3.0, -2.0])
true_b = 5.0
y = X @ true_w + true_b + np.random.randn(n) * 0.1

# Add bias column
X_b = np.c_[np.ones((n, 1)), X]  # (n, 3)

# Normal equation: w = (X^T X)^-1 X^T y
w = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(f"Learned bias: {w[0]:.3f} (true: 5.0)")
print(f"Learned w1: {w[1]:.3f} (true: 3.0)")
print(f"Learned w2: {w[2]:.3f} (true: -2.0)")

# Predictions and error
y_pred = X_b @ w
mse = np.mean((y - y_pred) ** 2)
print(f"\nMSE: {mse:.6f}")

Learned bias: 4.991 (true: 5.0)
Learned w1: 3.007 (true: 3.0)
Learned w2: -1.995 (true: -2.0)

MSE: 0.009773


## 12. Common Mistakes: Non-Conformable Matrices

The most common error in matrix multiplication: **inner dimensions don't match**.

```
A (1×3) @ B (2×2)  →  FAIL!  inner dims: 3 ≠ 2

The rule:
  A (m×n) @ B (n×p) = C (m×p)
  A (m×k) @ B (j×p) → ERROR if k ≠ j
```

**How to debug:** Always check `.shape` before multiplying. Print `A.shape` and `B.shape` and verify the middle two numbers match.

In [9]:
# Non-conformable matrices
A = np.array([[1, 2, 3]])   # 1x3
B = np.array([[1, 2], [3, 4]])  # 2x2

try:
    result = A @ B
except ValueError as e:
    print(f"ValueError: {e}")
    print(f"A shape: {A.shape}, B shape: {B.shape}")
    print("Inner dimensions must match: A.shape[1] == B.shape[0]")

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)
A shape: (1, 3), B shape: (2, 2)
Inner dimensions must match: A.shape[1] == B.shape[0]


## 13. Hands-On Practice

**Level 1 — Recall:** What are the shape rules for matrix multiplication? If A is (3×4) and B is (4×2), what is the shape of A @ B?

**Level 2 — Apply:** Multiply these two matrices by hand, then verify with NumPy:
```
A = [[1, 2],     B = [[5, 6],
     [3, 4]]          [7, 8]]
```

**Level 3 — Compute:** Solve the system:
```
2x + y = 5
x + 3y = 10
```
Verify by multiplying A @ x.

**Level 4 — Debug:** This code produces unexpected results. Why?
```python
A = np.array([[1, 2], [3, 4]])
b = np.array([5, 6])
x = np.linalg.inv(A) @ b  # works but might be unstable
```
What's the better approach? Why?

**Level 5 — Create:** Implement ridge regression from scratch: `w = (XᵀX + λI)⁻¹Xᵀy`. Show that as λ increases, the weights shrink toward zero.

## 14. Real-World ML Application

**Neural network forward pass** — a single layer:
```python
def forward(X, W, b):
    return np.maximum(0, X @ W + b)  # ReLU activation
```
The `X @ W` is matrix multiplication: (n_samples × n_features) @ (n_features × n_hidden).

**Normal equation** for linear regression:
```python
X_b = np.c_[np.ones((n,1)), X]  # add bias column
w = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
```

**Ridge regression** — adds regularization to prevent overfitting:
```python
lambda_reg = 0.1
w = np.linalg.solve(X_b.T @ X_b + lambda_reg * np.eye(X_b.shape[1]), X_b.T @ y)
```

## 15. Debugging: Common Errors

**`*` vs `@`**:
```python
A * B   # element-wise (Hadamard product) — shape must match exactly
A @ B   # matrix multiply — inner dimensions must match
```

**Singular matrix error**:
```python
np.linalg.inv(np.array([[1,2],[2,4]]))  # LinAlgError: Singular matrix
# Fix: check determinant, use solve, or add regularization
```

**Numerical instability**:
```python
# BAD: computing inverse explicitly
x = np.linalg.inv(A) @ b

# GOOD: using solve
x = np.linalg.solve(A, b)
```

## 16. Knowledge Check

Answer without running code:

1. What is the shape of `np.ones((3,4)) @ np.ones((4,2))`?
2. Why is `A @ A⁻¹ = I` but `A @ A⁻¹ ≠ A⁻¹ @ A` in general?
3. When does a matrix have no inverse? What's it called?
4. Why should you prefer `np.linalg.solve` over `np.linalg.inv(A) @ b`?
5. In the normal equation, what does the bias column (column of 1s) do?

## 17. Exit Criteria

You're ready to move on when you can:
- [ ] Multiply matrices by hand and predict the shape of the result
- [ ] Explain why matrix multiplication is not commutative (A @ B ≠ B @ A)
- [ ] Use `np.linalg.solve` to solve Ax = b
- [ ] Explain when a matrix is singular and what that means geometrically
- [ ] Implement the normal equation for linear regression
- [ ] Know when to use solve vs. inv vs. gradient descent

## 18. Next Step

**→ 02.3 Derivatives & Gradients**: Now that you can do matrix operations, learn the calculus that tells you *how to update* the weights — derivatives and gradients.